# 04 · Análise: correlações e teste das proposições (etapa 4)

Lê `dados/processados/base_diaria.csv` e gera, em `resultados/` e `figuras/`:
- correlação móvel de 63 pregões (S&P 500 x Treasury; Ibovespa x NTN-B) e a figura central;
- P1: correlação em crise x normal nos EUA (z de Fisher e bootstrap em blocos);
- P2: regressão da correlação móvel sobre o VIX (erros de Newey-West), nos dois países;
- P3: correlação Ibovespa x NTN-B em crise, com e sem controle pelo câmbio;
- P4: correlação em "só crise local" x "só crise global";
- 2022: descrição;
- robustez e `resultados/log-resultados.md`.

Convenção: "correlação no regime" é a correlação de Pearson dos retornos diários dos dias do regime.
A correlação móvel serve para a figura e para a P2.

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

RAIZ = Path.cwd().parent if Path.cwd().name == "codigo" else Path.cwd()
PASTA_RESULTADOS = RAIZ / "resultados"
PASTA_FIGURAS = RAIZ / "figuras"
PASTA_RESULTADOS.mkdir(exist_ok=True)
PASTA_FIGURAS.mkdir(exist_ok=True)

JANELA = 63            # pregões (principal)
JANELAS_ROBUSTEZ = [21, 126]
N_BOOT = 2000          # réplicas do bootstrap
BLOCO = 21             # tamanho do bloco (pregões) no bootstrap em blocos circular
BLOCO_ROBUSTEZ = 63
SEMENTE = 20261102
LAGS_NW = JANELA       # defasagens de Newey-West: a correlação móvel se sobrepõe por JANELA-1 dias

base = pd.read_csv(RAIZ / "dados" / "processados" / "base_diaria.csv", parse_dates=["data"], index_col="data")
base = base.iloc[1:].copy()  # 1ª linha não tem retorno
for c in [c for c in base.columns if base[c].dropna().isin([0, 1]).all() and c.startswith(("crise", "so_", "ambas", "normal", "choque"))]:
    base[c] = base[c].astype(bool)

# Regimes derivados usados na análise
base["normal_eua"] = ~base["crise_p1_eua"]
base["crise_br"] = base["crise_local_br"] | base["crise_global"]          # qualquer crise relevante para o Brasil
base["so_global_p90"] = base["crise_global_p90"] & ~base["crise_local_br"]
base["normal_p90"] = ~base["crise_local_br"] & ~base["crise_global_p90"]
base["crise_br_p90"] = base["crise_local_br"] | base["crise_global_p90"]
base["crise_p1_eua_p90"] = base["crise_global_p90"] | base["crise_eua"]
base["normal_eua_p90"] = ~base["crise_p1_eua_p90"]
base["crise_p1_vix"] = base["crise_global"]                                 # só VIX, sem drawdown
base["normal_eua_vix"] = ~base["crise_global"]
print(f"Base: {base.index.min().date()} a {base.index.max().date()}, {len(base)} dias com retorno")

Base: 2003-09-17 a 2026-09-30, 5525 dias com retorno


## Funções

In [2]:
def corr(x, y):
    return float(np.corrcoef(x, y)[0, 1])


def corr_parcial(x, y, z):
    """Correlação entre x e y depois de tirar o efeito linear de z (pode ter várias colunas)."""
    Z = sm.add_constant(np.column_stack([z]))
    ex = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
    ey = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return corr(ex, ey)


def fisher_z(r1, n1, r2, n2):
    """Teste z de Fisher (amostras independentes) para r1 - r2. Retorna z e p bicaudal."""
    z = (np.arctanh(r1) - np.arctanh(r2)) / np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    return float(z), float(2 * stats.norm.sf(abs(z)))


def ic_fisher(r, n, nivel=0.95):
    q = stats.norm.ppf(0.5 + nivel / 2)
    return float(np.tanh(np.arctanh(r) - q / np.sqrt(n - 3))), float(np.tanh(np.arctanh(r) + q / np.sqrt(n - 3)))


def indices_bootstrap(n, bloco, n_boot, semente):
    """Bootstrap em blocos circular (Politis e Romano): cada réplica junta blocos de dias consecutivos."""
    rng = np.random.default_rng(semente)
    n_blocos = int(np.ceil(n / bloco))
    inicios = rng.integers(0, n, size=(n_boot, n_blocos))
    idx = (inicios[:, :, None] + np.arange(bloco)[None, None, :]) % n
    return idx.reshape(n_boot, -1)[:, :n]


def corr_regime_boot(x, y, regime, idx, z=None):
    """Correlação (ou parcial, se z) dentro do regime em cada réplica. Vetorizado por réplica."""
    saida = np.full(idx.shape[0], np.nan)
    for b in range(idx.shape[0]):
        sel = idx[b][regime[idx[b]]]
        if len(sel) < 10:
            continue
        saida[b] = corr(x[sel], y[sel]) if z is None else corr_parcial(x[sel], y[sel], z[sel])
    return saida


def forbes_rigobon(r_crise, var_crise, var_normal):
    """Ajuste de Forbes e Rigobon (2002): rho* = rho / sqrt(1 + delta (1 - rho^2)), delta = var_crise/var_normal - 1."""
    delta = var_crise / var_normal - 1
    return float(r_crise / np.sqrt(1 + delta * (1 - r_crise ** 2)))


def comparar(base, x, y, regime_a, regime_b, controle=None, bloco=BLOCO, rotulo="", semente=SEMENTE, ajuste_fr=True):
    """Compara a correlação de x e y no regime A (crise) com o regime B (referência)."""
    cols = [x, y] + ([controle] if controle else [])
    d = base[cols + [regime_a, regime_b]].dropna()
    X, Y = d[x].to_numpy(), d[y].to_numpy()
    Z = d[controle].to_numpy() if controle else None
    A, B = d[regime_a].to_numpy(bool), d[regime_b].to_numpy(bool)
    f = (lambda m: corr(X[m], Y[m])) if Z is None else (lambda m: corr_parcial(X[m], Y[m], Z[m]))
    ra, rb, na, nb = f(A), f(B), int(A.sum()), int(B.sum())
    z, p = fisher_z(ra, na, rb, nb)
    idx = indices_bootstrap(len(d), bloco, N_BOOT, semente)
    ba = corr_regime_boot(X, Y, A, idx, Z)
    bb = corr_regime_boot(X, Y, B, idx, Z)
    dif = ba - bb
    linha = {
        "teste": rotulo, "x": x, "y": y, "controle": controle or "", "regime_a": regime_a, "regime_b": regime_b,
        "n_a": na, "n_b": nb, "r_a": ra, "r_b": rb, "dif": ra - rb,
        "ic95_r_a_boot": (np.nanpercentile(ba, 2.5), np.nanpercentile(ba, 97.5)),
        "ic95_dif_boot": (np.nanpercentile(dif, 2.5), np.nanpercentile(dif, 97.5)),
        "p_boot_dif": float(min(1, 2 * min(np.nanmean(dif <= 0), np.nanmean(dif >= 0)))),
        "z_fisher": z, "p_fisher": p, "bloco": bloco,
    }
    if ajuste_fr and Z is None:
        linha["r_a_forbes_rigobon"] = forbes_rigobon(ra, X[A].var(), X[B].var())
    return linha


def fmt_ic(ic):
    return f"[{ic[0]:.3f}; {ic[1]:.3f}]"

## Correlação móvel e figura central

In [3]:
PARES = {
    "eua": ("r_sp500", "r_treasury", "S&P 500 x Treasury 10 anos"),
    "br": ("r_ibov", "r_ntnb", "Ibovespa x NTN-B (IMA-B)"),
}
moveis = pd.DataFrame(index=base.index)
for janela in [JANELA] + JANELAS_ROBUSTEZ:
    for pais, (x, y, _) in PARES.items():
        moveis[f"corr{janela}_{pais}"] = base[x].rolling(janela, min_periods=janela).corr(base[y])
moveis["vix"] = base["vix"]
moveis["vix_medio63"] = base["vix"].rolling(JANELA, min_periods=JANELA).mean()
moveis.to_csv(PASTA_RESULTADOS / "correlacoes_moveis.csv", float_format="%.6f")


def sombrear(ax, indicador, cor, rotulo):
    ind = indicador.astype(int)
    grupos = (ind != ind.shift()).cumsum()
    primeiro = True
    for _, g in ind[ind == 1].groupby(grupos[ind == 1]):
        ax.axvspan(g.index[0], g.index[-1], color=cor, alpha=0.25, lw=0, label=rotulo if primeiro else None)
        primeiro = False


fig, eixos = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax = eixos[0]
sombrear(ax, base["crise_p1_eua"], "#d62728", "Crise EUA (VIX > 30 ou drawdown do S&P 500 ≤ −20%)")
ax.plot(moveis.index, moveis[f"corr{JANELA}_eua"], color="#1f3b73", lw=1)
ax.axhline(0, color="black", lw=0.6)
ax.set_title("(a) EUA: S&P 500 x Treasury 10 anos")
ax.set_ylabel("Correlação (63 pregões)")
ax.set_ylim(-1, 1)
ax.legend(loc="lower left", fontsize=8, frameon=False)
ax = eixos[1]
sombrear(ax, base["so_local"], "#ff7f0e", "Só crise local (drawdown do Ibovespa ≤ −20%)")
sombrear(ax, base["so_global"], "#2ca02c", "Só crise global (VIX > 30)")
sombrear(ax, base["ambas"], "#d62728", "Crise local e global")
ax.plot(moveis.index, moveis[f"corr{JANELA}_br"], color="#1f3b73", lw=1)
ax.axhline(0, color="black", lw=0.6)
ax.set_title("(b) Brasil: Ibovespa x NTN-B (IMA-B)")
ax.set_ylabel("Correlação (63 pregões)")
ax.set_ylim(-1, 1)
ax.legend(loc="lower left", fontsize=8, frameon=False, ncol=3)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.text(0.01, 0.005, "Fonte: elaborado pelos autores com dados de Yahoo Finance, Federal Reserve (H.15), Cboe e ANBIMA.", fontsize=8)
fig.tight_layout(rect=(0, 0.02, 1, 1))
fig.savefig(PASTA_FIGURAS / "figura1_correlacao_movel.png", dpi=200)
plt.close(fig)
print("figuras/figura1_correlacao_movel.png")

media_regime = pd.DataFrame({
    "EUA (63d)": [moveis.loc[base[r], f"corr{JANELA}_eua"].mean() for r in ["normal", "so_local", "so_global", "ambas", "crise_p1_eua"]],
    "Brasil (63d)": [moveis.loc[base[r], f"corr{JANELA}_br"].mean() for r in ["normal", "so_local", "so_global", "ambas", "crise_p1_eua"]],
}, index=["normal", "so_local", "so_global", "ambas", "crise_p1_eua"]).round(3)
print(media_regime)

figuras/figura1_correlacao_movel.png
              EUA (63d)  Brasil (63d)
normal           -0.218         0.345
so_local         -0.344         0.304
so_global        -0.292         0.386
ambas            -0.449         0.344
crise_p1_eua     -0.349         0.332


## P1: EUA, crise x período normal

In [4]:
testes = []
p1 = comparar(base, "r_sp500", "r_treasury", "crise_p1_eua", "normal_eua", rotulo="P1 principal")
testes.append(p1)
print({k: p1[k] for k in ["n_a", "n_b", "r_a", "r_b", "dif", "ic95_dif_boot", "p_boot_dif", "p_fisher", "r_a_forbes_rigobon"]})

{'n_a': 524, 'n_b': 5001, 'r_a': -0.36052626018548867, 'r_b': -0.24081459160876847, 'dif': -0.11971166857672019, 'ic95_dif_boot': (np.float64(-0.22461516563724013), np.float64(-0.002371850345359698)), 'p_boot_dif': 0.048, 'p_fisher': 0.004183242551712508, 'r_a_forbes_rigobon': -0.12356997035416549}


## P2: correlação móvel x VIX (Newey-West)

In [5]:
def regressao_vix(pais, janela=JANELA, futura=False, tendencia=False, log_vix=False):
    """corr_t = a + b·VIX_t (+ tendência). Se futura, a correlação é a dos próximos 22 pregões (como Connolly et al.)."""
    x, y, _ = PARES[pais]
    if futura:
        c = base[x].rolling(22).corr(base[y]).shift(-22)
        lags = 22
    else:
        c = moveis[f"corr{janela}_{pais}"]
        lags = janela
    v = np.log(base["vix"]) if log_vix else base["vix"]
    d = pd.DataFrame({"c": c, "vix": v}).dropna()
    X = d[["vix"]]
    if tendencia:
        X = X.assign(tendencia=np.arange(len(d)) / 252)
    m = sm.OLS(d["c"], sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return {"pais": pais, "janela": 22 if futura else janela, "futura": futura, "tendencia": tendencia, "log_vix": log_vix,
            "n": int(m.nobs), "b_vix": float(m.params["vix"]), "ep_nw": float(m.bse["vix"]), "t": float(m.tvalues["vix"]),
            "p": float(m.pvalues["vix"]), "r2": float(m.rsquared)}


p2 = [regressao_vix(p) for p in ["eua", "br"]]
p2_rob = []
for p in ["eua", "br"]:
    for kw in [dict(futura=True), dict(tendencia=True), dict(log_vix=True), dict(janela=21), dict(janela=126)]:
        p2_rob.append(regressao_vix(p, **kw))
# Diferença entre países: a correlação do Brasil reage ao VIX diferente da dos EUA?
d = pd.DataFrame({"eua": moveis[f"corr{JANELA}_eua"], "br": moveis[f"corr{JANELA}_br"], "vix": base["vix"]}).dropna()
dd = (d["br"] - d["eua"])
m_dif = sm.OLS(dd, sm.add_constant(d["vix"])).fit(cov_type="HAC", cov_kwds={"maxlags": JANELA})
p2_dif = {"b_vix_br_menos_eua": float(m_dif.params["vix"]), "p": float(m_dif.pvalues["vix"])}
print(pd.DataFrame(p2).round(4))
print(p2_dif)

  pais  janela  futura  tendencia  log_vix     n   b_vix   ep_nw       t  \
0  eua      63   False      False    False  5463 -0.0078  0.0021 -3.8011   
1   br      63   False      False    False  5463  0.0023  0.0025  0.9065   

        p      r2  
0  0.0001  0.0431  
1  0.3647  0.0057  
{'b_vix_br_menos_eua': 0.010071625032248238, 'p': 0.00150662911288838}


## P3: Brasil, título doméstico em crise, com controle do câmbio

In [6]:
p3 = comparar(base, "r_ibov", "r_ntnb", "crise_br", "normal", rotulo="P3 sem controle")
p3c = comparar(base, "r_ibov", "r_ntnb", "crise_br", "normal", controle="r_usdbrl", rotulo="P3 controle PTAX")
p3f = comparar(base, "r_ibov", "r_ntnb", "crise_br", "normal", controle="r_usdbrl_fech", rotulo="P3 controle BRL=X")
testes += [p3, p3c, p3f]


def regressao_p3(regime, controle="r_usdbrl"):
    d = base.loc[base[regime], ["r_ntnb", "r_ibov", controle]].dropna()
    m = sm.OLS(d["r_ntnb"], sm.add_constant(d[["r_ibov", controle]])).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return {"regime": regime, "controle": controle, "n": int(m.nobs), "b_ibov": float(m.params["r_ibov"]),
            "p_ibov": float(m.pvalues["r_ibov"]), "b_cambio": float(m.params[controle]), "p_cambio": float(m.pvalues[controle]),
            "r2": float(m.rsquared)}


p3_reg = [regressao_p3(r) for r in ["crise_br", "normal", "so_local", "so_global", "ambas"]]
print(pd.DataFrame(p3_reg).round(4))

      regime  controle     n  b_ibov  p_ibov  b_cambio  p_cambio      r2
0   crise_br  r_usdbrl   840  0.0892  0.0004   -0.0371    0.0526  0.1916
1     normal  r_usdbrl  4685  0.0783  0.0000   -0.1124    0.0000  0.2021
2   so_local  r_usdbrl   418  0.0548  0.0065   -0.0111    0.7619  0.0331
3  so_global  r_usdbrl   111  0.0695  0.0000   -0.0945    0.1613  0.2607
4      ambas  r_usdbrl   311  0.0999  0.0024   -0.0398    0.0828  0.3167


## P4: só crise local x só crise global

In [7]:
p4 = comparar(base, "r_ibov", "r_ntnb", "so_local", "so_global", rotulo="P4 principal")
p4c = comparar(base, "r_ibov", "r_ntnb", "so_local", "so_global", controle="r_usdbrl", rotulo="P4 controle PTAX")
testes += [p4, p4c]
# Cada um contra o normal, para ler o nível
for reg in ["so_local", "so_global", "ambas"]:
    testes.append(comparar(base, "r_ibov", "r_ntnb", reg, "normal", rotulo=f"BR {reg} x normal"))

## 2022 (exploratória)

In [8]:
anos = []
for ano in range(2004, 2027):
    d = base.loc[str(ano)]
    anos.append({"ano": ano, "n": len(d), "corr_eua": corr(d["r_sp500"], d["r_treasury"]), "corr_br": corr(d["r_ibov"], d["r_ntnb"]),
                 "vix_medio": d["vix"].mean(), "ret_sp500": d["r_sp500"].sum(), "ret_treasury": d["r_treasury"].sum(),
                 "ret_ibov": d["r_ibov"].sum(), "ret_ntnb": d["r_ntnb"].sum()})
anos = pd.DataFrame(anos)
anos.to_csv(PASTA_RESULTADOS / "correlacao_por_ano.csv", index=False, float_format="%.4f")
ex2022 = {
    "2022": comparar(base.loc["2018":"2023"].assign(a2022=lambda b: b.index.year == 2022, fora=lambda b: b.index.year != 2022),
                     "r_sp500", "r_treasury", "a2022", "fora", rotulo="EUA 2022 x 2018-2023 sem 2022", ajuste_fr=False),
}
ex2022_br = comparar(base.loc["2018":"2023"].assign(a2022=lambda b: b.index.year == 2022, fora=lambda b: b.index.year != 2022),
                     "r_ibov", "r_ntnb", "a2022", "fora", rotulo="BR 2022 x 2018-2023 sem 2022", ajuste_fr=False)
testes += [ex2022["2022"], ex2022_br]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(anos["ano"], anos["corr_eua"], marker="o", label="EUA: S&P 500 x Treasury")
ax.plot(anos["ano"], anos["corr_br"], marker="s", label="Brasil: Ibovespa x NTN-B")
ax.axhline(0, color="black", lw=0.6)
ax.axvspan(2021.5, 2022.5, color="grey", alpha=0.2, lw=0)
ax.set_ylabel("Correlação dos retornos diários no ano")
ax.set_title("Correlação ações x títulos por ano (2022 sombreado)")
ax.legend(frameon=False)
fig.text(0.01, 0.005, "Fonte: elaborado pelos autores com dados de Yahoo Finance, Federal Reserve (H.15) e ANBIMA. 2026 até 30/09.", fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(PASTA_FIGURAS / "figura2_correlacao_por_ano.png", dpi=200)
plt.close(fig)

## Robustez

In [9]:
rob = []
# P1
rob.append(comparar(base, "r_sp500", "r_treasury", "crise_p1_eua_p90", "normal_eua_p90", rotulo="P1 VIX p90"))
rob.append(comparar(base, "r_sp500", "r_treasury", "crise_p1_vix", "normal_eua_vix", rotulo="P1 só VIX > 30"))
rob.append(comparar(base, "r_sp500", "r_tips", "crise_p1_eua", "normal_eua", rotulo="P1 TIPS"))
rob.append(comparar(base, "r_sp500", "r_ief", "crise_p1_eua", "normal_eua", rotulo="P1 IEF"))
rob.append(comparar(base, "r_sp500", "r_treasury", "crise_p1_eua", "normal_eua", bloco=BLOCO_ROBUSTEZ, rotulo="P1 bloco 63"))
# P3
for y, nome in [("r_ntnb5mais", "IMA-B 5+"), ("r_idka10", "IDkA IPCA 10A"), ("r_irfm", "IRF-M")]:
    rob.append(comparar(base, "r_ibov", y, "crise_br", "normal", controle="r_usdbrl", rotulo=f"P3 {nome} (controle PTAX)"))
rob.append(comparar(base, "r_ibov_usd", "r_ntnb_usd", "crise_br", "normal", rotulo="P3 retornos em USD"))
rob.append(comparar(base, "r_ibov", "r_ntnb", "crise_br_p90", "normal_p90", controle="r_usdbrl", rotulo="P3 VIX p90 (controle PTAX)"))
rob.append(comparar(base, "r_ibov", "r_ntnb", "crise_br", "normal", controle="r_usdbrl", bloco=BLOCO_ROBUSTEZ, rotulo="P3 bloco 63 (controle PTAX)"))
# P4
for reg_local, nome in [("so_local_p90", "VIX p90"), ("so_local_sem_cauda", "sem cauda de crise global (a)"),
                        ("crise_local_fixa", "lista fixa de eventos (b)"), ("so_local_choque", "choque de 21 pregões (c)")]:
    reg_global = "so_global_p90" if reg_local == "so_local_p90" else "so_global"
    rob.append(comparar(base, "r_ibov", "r_ntnb", reg_local, reg_global, rotulo=f"P4 {nome}"))
for y, nome in [("r_ntnb5mais", "IMA-B 5+"), ("r_irfm", "IRF-M")]:
    rob.append(comparar(base, "r_ibov", y, "so_local", "so_global", rotulo=f"P4 {nome}"))
rob.append(comparar(base, "r_ibov_usd", "r_ntnb_usd", "so_local", "so_global", rotulo="P4 retornos em USD"))
rob.append(comparar(base, "r_ibov", "r_ntnb", "so_local", "so_global", bloco=BLOCO_ROBUSTEZ, rotulo="P4 bloco 63"))


def tabela(linhas):
    t = pd.DataFrame(linhas)
    for c in ["ic95_r_a_boot", "ic95_dif_boot"]:
        t[c] = t[c].map(fmt_ic)
    return t


t_testes, t_rob = tabela(testes), tabela(rob)
t_testes.to_csv(PASTA_RESULTADOS / "testes_proposicoes.csv", index=False, float_format="%.4f")
t_rob.to_csv(PASTA_RESULTADOS / "robustez.csv", index=False, float_format="%.4f")
pd.DataFrame(p2 + p2_rob).to_csv(PASTA_RESULTADOS / "p2_regressoes_vix.csv", index=False, float_format="%.5f")
pd.DataFrame(p3_reg).to_csv(PASTA_RESULTADOS / "p3_regressoes.csv", index=False, float_format="%.5f")
media_regime.to_csv(PASTA_RESULTADOS / "correlacao_movel_media_por_regime.csv")
COLS = ["teste", "controle", "n_a", "n_b", "r_a", "r_b", "dif", "ic95_dif_boot", "p_boot_dif", "p_fisher", "r_a_forbes_rigobon"]
print(t_testes[COLS].round(3).to_string())
print(t_rob[COLS].round(3).to_string())

                            teste       controle  n_a   n_b    r_a    r_b    dif     ic95_dif_boot  p_boot_dif  p_fisher  r_a_forbes_rigobon
0                    P1 principal                 524  5001 -0.361 -0.241 -0.120  [-0.225; -0.002]       0.048     0.004              -0.124
1                 P3 sem controle                 840  4685  0.430  0.387  0.043   [-0.176; 0.214]       0.780     0.168               0.234
2                P3 controle PTAX       r_usdbrl  840  4685  0.364  0.298  0.067   [-0.144; 0.248]       0.687     0.046                 NaN
3               P3 controle BRL=X  r_usdbrl_fech  816  4223  0.419  0.393  0.026   [-0.202; 0.217]       0.890     0.421                 NaN
4                    P4 principal                 418   111  0.181  0.431 -0.250  [-0.429; -0.108]       0.001     0.010               0.186
5                P4 controle PTAX       r_usdbrl  418   111  0.163  0.323 -0.160   [-0.403; 0.000]       0.051     0.114                 NaN
6            